In [55]:
import os
import warnings
import pickle
import logging
import random
from copy import deepcopy as dc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

In [56]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

In [57]:
DATA_PATH       = "/kaggle/input/datasets/quanghuylt/stock-price-2/stock_price.csv"
METRICS_DIR     = "/kaggle/working/reports/metrics/gru/"
RESULTS_DIR     = "/kaggle/working/data/meta_test/"
FIGURES_DIR     = "/kaggle/working/reports/figures/gru"
MODEL_DIR       = "/kaggle/working/models/gru/"
OOF_DIR         = "/kaggle/working/data/meta_train/"
SUMMARY_METRICS = "/kaggle/working/reports/metrics/gru/GRU_all_tickers.csv"
os.makedirs(os.path.dirname(SUMMARY_METRICS), exist_ok=True)

SYMBOLS      = ["VNM", "FPT", "MSN", "VCB", "VIC", "HPG"]
TARGET_COL   = "Close"

TRAIN_RATIO  = 0.8
N_FOLDS      = 10
GAP_DAYS     = 10
LOOKBACK     = 7

HIDDEN_SIZE  = 32
NUM_LAYERS   = 2
BATCH_SIZE   = 16
EPOCHS       = 50
LR           = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [58]:
def split_data(df: pd.DataFrame):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    split_idx = int(len(df) * TRAIN_RATIO)
    train_df  = df.iloc[:split_idx].reset_index(drop=True)
    test_df   = df.iloc[split_idx:].reset_index(drop=True)
    return train_df, test_df

In [59]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc  = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.gru.num_layers, x.size(0), self.gru.hidden_size).to(x.device)
        out, _ = self.gru(x, h0)
        return self.fc(out[:, -1, :])

In [60]:
def build_xy_local(df: pd.DataFrame, n_steps: int):
    prices = df[TARGET_COL].values
    n = len(prices)
    X, Y = [], []
    mus, sigmas = [], []

    for i in range(n - n_steps):
        window_x = prices[i : i + n_steps] 
        target_y = prices[i + n_steps]
        mu = np.mean(window_x)
        sigma = np.std(window_x) + 1e-8
        scaled_x = (window_x - mu) / sigma
        scaled_y = (target_y - mu) / sigma
        X.append(scaled_x)
        Y.append(scaled_y)
        mus.append(mu)
        sigmas.append(sigma)

    X_t = torch.tensor(np.array(X).reshape(-1, n_steps, 1), dtype=torch.float32)
    Y_t = torch.tensor(np.array(Y).reshape(-1, 1), dtype=torch.float32)
    return X_t, Y_t, np.array(mus), np.array(sigmas)

In [61]:
def train_model(X_t: torch.Tensor, Y_t: torch.Tensor) -> GRUModel:
    dataset = TensorDataset(X_t, Y_t)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    model     = GRUModel(input_size=1, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, output_size=1).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()
    for epoch in range(EPOCHS):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        if (epoch + 1) % 10 == 0:
            log.info(f"      epoch {epoch+1}/{EPOCHS}  loss={loss.item():.6f}")
    return model

In [62]:
def run_oof(train_df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    n = len(train_df)
    fold_size = n // (N_FOLDS + 1)
    oof_preds = np.full(n, np.nan)
    log.info(f"    OOF: {N_FOLDS} folds, fold_size={fold_size}, gap={GAP_DAYS}, lookback={LOOKBACK}")
    for k in range(1, N_FOLDS + 1):
        train_end = k * fold_size
        val_start = train_end + GAP_DAYS
        val_end   = (k + 1) * fold_size if k < N_FOLDS else n
        if val_start + LOOKBACK >= val_end: continue
        fold_train = train_df.iloc[:train_end].copy()
        fold_val   = train_df.iloc[val_start:val_end].copy()
        try:
            X_tr, Y_tr, _, _ = build_xy_local(fold_train, LOOKBACK)
            model = train_model(X_tr, Y_tr)
            context = list(fold_train[TARGET_COL].values[-LOOKBACK:])
            fold_preds = []
            model.eval()
            with torch.no_grad():
                for i in range(len(fold_val)):
                    window = np.array(context[-LOOKBACK:])
                    mu, sigma = np.mean(window), np.std(window) + 1e-8
                    scaled_window = (window - mu) / sigma
                    x = torch.tensor(scaled_window.reshape(1, LOOKBACK, 1), dtype=torch.float32).to(DEVICE)
                    y_real = model(x).cpu().numpy().flatten()[0] * sigma + mu
                    fold_preds.append(y_real)
                    context.append(fold_val[TARGET_COL].values[i]) 
            fill_len = min(len(fold_preds), val_end - val_start)
            oof_preds[val_start:val_start + fill_len] = fold_preds[:fill_len]
            log.info(f"      Fold {k}: val=[{val_start}:{val_end}] MAE={mean_absolute_error(fold_val[TARGET_COL].values[:fill_len], fold_preds[:fill_len]):.4f}")
            del model; torch.cuda.empty_cache()
        except Exception as e: log.warning(f"      Fold {k} failed: {e}")
    return pd.DataFrame({"Ticker": ticker, "Date": train_df["Date"].values, "Close": train_df[TARGET_COL].values, "GRU_Predicted_Close": oof_preds})

In [63]:
def predict_test(model: GRUModel, train_df: pd.DataFrame, test_df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    model.eval()
    context = list(train_df[TARGET_COL].values[-LOOKBACK:])
    preds_real = []
    with torch.no_grad():
        for i in range(len(test_df)):
            window = np.array(context[-LOOKBACK:])
            mu, sigma = np.mean(window), np.std(window) + 1e-8
            x = torch.tensor(((window - mu) / sigma).reshape(1, LOOKBACK, 1), dtype=torch.float32).to(DEVICE)
            y_real = model(x).cpu().numpy().flatten()[0] * sigma + mu
            preds_real.append(y_real)
            context.append(test_df[TARGET_COL].values[i]) 
    return pd.DataFrame({"Ticker": ticker, "Date": test_df["Date"].values, "Close": test_df[TARGET_COL].values, "GRU_Predicted_Close": np.array(preds_real)})

In [64]:
def calc_metrics(
    df: pd.DataFrame,
    actual_col: str = "Close",
    pred_col:   str = "GRU_Predicted_Close",
    ticker_col: str = "Ticker",
) -> pd.DataFrame:

    if ticker_col and ticker_col in df.columns and df[ticker_col].nunique() > 1:
        return (
            df.groupby(ticker_col)
              .apply(lambda x: calc_metrics(x, actual_col, pred_col, ticker_col=None))
              .reset_index()
        )

    df = df.copy()
    df = df.dropna(subset=[actual_col, pred_col])
    df["Prev_Actual"] = df[actual_col].shift(1)
    df["Prev_Pred"]   = df[pred_col].shift(1)
    df = df.dropna(subset=["Prev_Actual", "Prev_Pred"])

    y_true = df[actual_col]
    y_pred = df[pred_col]

    mse  = mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8)))
    r2   = r2_score(y_true, y_pred)

    actual_dir = np.sign(y_true.values - df["Prev_Actual"].values)
    pred_dir   = np.sign(y_pred.values - df["Prev_Pred"].values)
    da         = np.mean(actual_dir == pred_dir)

    mask  = actual_dir != 0
    tpa   = np.mean(actual_dir[mask] == pred_dir[mask]) if np.any(mask) else np.nan

    actual_vol = y_true - y_true.mean()
    pred_vol   = y_pred - y_pred.mean()
    v_rmse     = np.sqrt(mean_squared_error(actual_vol, pred_vol))

    return pd.DataFrame([{
        "MSE": mse, "MAE": mae, "MAPE": mape, "RMSE": rmse,
        "R2": r2, "DA": da, "TPA": tpa, "V-RMSE": v_rmse,
    }])

In [65]:
def plot_results(df: pd.DataFrame, save_dir: str) -> None:
    ticker, df_plot = df["Ticker"].iloc[0], df.dropna(subset=["GRU_Predicted_Close"])
    os.makedirs(save_dir, exist_ok=True); sns.set(style="whitegrid")
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    axes[0].plot(df_plot["Date"], df_plot["Close"], label="Actual", color="#1f77b4")
    axes[0].plot(df_plot["Date"], df_plot["GRU_Predicted_Close"], label="GRU", color="#d62728", linestyle="--")
    axes[0].set_title(f"GRU Results: {ticker}"); axes[0].legend()
    error = df_plot["Close"] - df_plot["GRU_Predicted_Close"]
    axes[1].bar(df_plot["Date"], error, color=np.where(error >= 0, "#2ca02c", "#d62728"))
    axes[1].set_title(f"Error: {ticker}"); plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{ticker}.png"), dpi=150); plt.close()

In [ ]:
def train_one_ticker(ticker: str) -> pd.DataFrame:
    log.info(f"========== {ticker} ==========")
    
    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df = df[df["Ticker"] == ticker].sort_values("Date").reset_index(drop=True)
    
    train_df, test_df = split_data(df)
    log.info(f"  Train: {len(train_df)} rows  Test: {len(test_df)} rows")
    
    oof_df = run_oof(train_df, ticker)
    
    log.info("    Training final model on full train set...")
    X_tr, Y_tr, mus, sigs = build_xy_local(train_df, LOOKBACK)
    final_model = train_model(X_tr, Y_tr)
    
    test_res = predict_test(final_model, train_df, test_df, ticker)
    
    final_model.eval()
    with torch.no_grad(): 
        preds_real = final_model(X_tr.to(DEVICE)).cpu().numpy().flatten() * sigs + mus
        
    train_refit_df = pd.DataFrame({
        "Ticker": ticker, 
        "Date": train_df["Date"].values[LOOKBACK:],
        "Close": train_df[TARGET_COL].values[LOOKBACK:], 
        "GRU_Predicted_Close": preds_real
    })
    
    refit_metrics = calc_metrics(train_refit_df)
    refit_metrics.insert(0, "Ticker", ticker)
    log.info(f"  Refit Train — MAE={refit_metrics['MAE'].iloc[0]:.2f} DA={refit_metrics['DA'].iloc[0]:.2%} R2={refit_metrics['R2'].iloc[0]:.4f}")
    
    metrics_df = calc_metrics(oof_df)
    metrics_df.insert(0, "Ticker", ticker)
    log.info(f"  OOF  — MAE={metrics_df['MAE'].iloc[0]:.2f} DA={metrics_df['DA'].iloc[0]:.2%} R2={metrics_df['R2'].iloc[0]:.4f}")
    
    test_metrics = calc_metrics(test_res)
    test_metrics.insert(0, "Ticker", ticker)
    log.info(f"  Test — MAE={test_metrics['MAE'].iloc[0]:.2f} DA={test_metrics['DA'].iloc[0]:.2%} R2={test_metrics['R2'].iloc[0]:.4f}")
    
    os.makedirs(METRICS_DIR, exist_ok=True)
    metrics_df.to_csv(os.path.join(METRICS_DIR, f"GRU_{ticker}_metrics.csv"), index=False)
    
    os.makedirs(OOF_DIR, exist_ok=True)
    oof_df.to_csv(os.path.join(OOF_DIR, f"oof_gru_{ticker}.csv"), index=False)
    
    os.makedirs(RESULTS_DIR, exist_ok=True)
    test_res.to_csv(os.path.join(RESULTS_DIR, f"GRU_{ticker}_predictions.csv"), index=False)
    
    plot_results(test_res, FIGURES_DIR)
    
    os.makedirs(MODEL_DIR, exist_ok=True)
    with open(os.path.join(MODEL_DIR, f"gru_{ticker}.pkl"), "wb") as f:
        pickle.dump({
            "model_state":  final_model.state_dict(),
            "ticker":       ticker
        }, f)
        
    torch.cuda.empty_cache()
    log.info(f"  All outputs saved for {ticker}")
    return test_metrics

In [67]:
all_metrics = []

for ticker in SYMBOLS:
    try: 
        m = train_one_ticker(ticker)
        all_metrics.append(m)
    except Exception as e: 
        log.error(f"{ticker} FAILED: {e}")

if all_metrics:
    summary_df = pd.concat(all_metrics, ignore_index=True)
    os.makedirs(os.path.dirname(SUMMARY_METRICS) or ".", exist_ok=True)
    summary_df.to_csv(SUMMARY_METRICS, index=False)
    log.info(f"Summary saved: {SUMMARY_METRICS}")
    print("\n" + summary_df.to_string(index=False))

log.info("Done.")

01:10:36  INFO      ========== VNM ==========
01:10:36  INFO        Train: 3259 rows  Test: 815 rows
01:10:36  INFO          OOF: 10 folds, fold_size=296, gap=10, lookback=7
01:10:36  INFO            epoch 10/50  loss=1.743302
01:10:37  INFO            epoch 20/50  loss=0.007536
01:10:37  INFO            epoch 30/50  loss=1.433724
01:10:38  INFO            epoch 40/50  loss=0.121270
01:10:38  INFO            epoch 50/50  loss=0.577064
01:10:38  INFO            Fold 1: val=[306:592] MAE=0.1335
01:10:39  INFO            epoch 10/50  loss=4.850756
01:10:40  INFO            epoch 20/50  loss=1.400862
01:10:41  INFO            epoch 30/50  loss=0.918764
01:10:42  INFO            epoch 40/50  loss=2.170203
01:10:43  INFO            epoch 50/50  loss=3.296481
01:10:43  INFO            Fold 2: val=[602:888] MAE=0.2938
01:10:44  INFO            epoch 10/50  loss=0.000098
01:10:45  INFO            epoch 20/50  loss=0.566711
01:10:47  INFO            epoch 30/50  loss=12.140492
01:10:48  INFO    


Ticker      MSE      MAE     MAPE     RMSE       R2       DA      TPA   V-RMSE
   VNM 2.547514 1.108598 0.018346 1.596094 0.832801 0.436118 0.469577 1.594751
   FPT 2.858021 1.176361 0.012940 1.690568 0.994725 0.508600 0.531451 1.675356
   MSN 3.492321 1.332679 0.018091 1.868775 0.930123 0.460688 0.496032 1.867207
   VCB 0.786873 0.584331 0.009804 0.887058 0.952595 0.449631 0.489960 0.887055
   VIC 6.521664 1.192758 0.017606 2.553755 0.997064 0.511057 0.573793 2.548570
   HPG 0.177651 0.290529 0.013015 0.421487 0.985691 0.447174 0.482119 0.421372
